## Function Calling (Tool Use)

**Week 09 — Revision Notebook**

এই notebook-এ ধাপে ধাপে দেখি, LLM কীভাবে আমার Python function চালাতে বলে।
প্রতিটা ধাপের আগে ছোট ব্যাখ্যা, তারপর code, নিচে output।
উপর থেকে নিচে **Run All** করলে কোনো error আসবে না।

**পুরো ছবি (পাঁচ লাইনে):**
1. আমি প্রশ্ন আর `tools`-এর তালিকা পাঠাই।
2. LLM ঠিক করে কোন tool লাগবে। সে নিজে function চালায় না, শুধু নাম আর argument বলে।
3. আমার code ওই function চালায়।
4. Result `messages`-এ যোগ করে আবার LLM-কে পাঠাই।
5. LLM result দেখে বাংলায় চূড়ান্ত উত্তর লেখে।

### সূচিপত্র (Table of Contents)

- [Topic ১: Function Calling কী](#t1)
- [ধাপ ১: import ও client](#s1)
- [ধাপ ২: ORDERS ও get_order_status](#s2)
- [ধাপ ৩: INVENTORY ও get_inventory](#s3)
- [ধাপ ৪: tools list (দুটো tool definition)](#s4)
  - [Topic ২: Tool Definition](#t2)
  - [Topic ৩: JSON Schema ও Parameter Design](#t3)
- [ধাপ ৫: LLM-এর বাছাই দেখা](#s5)
  - [Topic ৪: Tool Selection ও Tool Response](#t4)
- [ধাপ ৬: arguments পড়া ও routing](#s6)
- [ধাপ ৭: result ফেরত দিয়ে চূড়ান্ত উত্তর](#s7)
- [ধাপ ৮: run_tool function](#s8)
- [ধাপ ৯: দুই অংশের প্রশ্ন (sequential tool calls)](#s9)
- [ধাপ ১০: আমার করা ভুলগুলো ও self-check](#s10)
  - [আমার করা ভুলগুলো](#s11)
  - [Self-check](#s12)

<a id="t1"></a>

#### Topic ১: Function Calling কী

**আগে একটা গল্প**

পাড়ার চায়ের দোকানে একজন দোকানদার আছেন, যিনি অনেক কিছু জানেন। দেশের রাজধানী, ক্রিকেটের ইতিহাস, চা বানানোর নিয়ম, সব তাঁর মুখস্থ। কিন্তু তিনি কাউন্টার ছেড়ে নড়তে পারেন না। একদিন একজন খদ্দের জিজ্ঞেস করল, "ভাই, গুদামে আর কত কেজি চিনি আছে?" দোকানদার নিজে গুদামে যেতে পারেন না, আর তাঁর মুখস্থ জ্ঞানেও আজকের চিনির হিসাব নেই। তাই তিনি পাশের ছেলেটাকে বলেন, "গুদামে গিয়ে চিনি গুনে এসো।" ছেলেটা গুনে এসে সংখ্যাটা বলে, আর দোকানদার সেটা খদ্দেরকে সুন্দর করে জানিয়ে দেন।

**মূল ধারণা**

LLM হলো ওই দোকানদার। সে অনেক কিছু জানে, কিন্তু তার জ্ঞান পুরোনো, আর সে বাইরের জগৎ ছুঁতে পারে না। ছেলেটা হলো আমাদের backend code। Function Calling মানে LLM-কে আগে থেকে জানিয়ে রাখা যে কোন কোন কাজ করানো যায়। তারপর দরকার মনে করলে LLM বলে দেয়, "এই function-টা চালাও"। চালানোর কাজটা সবসময় আমাদের code-এর।

**মনে রাখার এক লাইন**

LLM শুধু বলে কোন function চালাতে হবে, চালায় আমাদের code।

<a id="s1"></a>

## ধাপ ১: import ও client

**ধাপটা কী:** দরকারি package আনা আর LLM-এর সাথে কথা বলার `client` বানানো।
- `json`: LLM-এর দেওয়া arguments পড়তে লাগবে।
- `os` আর `load_dotenv`: API key পড়তে। Key `.env` file-এ থাকে, code-এ লেখা থাকে না।
- `OpenAI`: Groq-এর URL দিলে এই client Groq-এর সাথেও চলে।

**কেন লাগে:** client ছাড়া LLM-কে প্রশ্ন পাঠানো যায় না। `.env` কখনো commit করা যাবে না (`.gitignore`-এ আছে)।

In [31]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = "openai/gpt-oss-20b"

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

print("API key পাওয়া গেছে:", os.getenv("GROQ_API_KEY") is not None)
print("model:", MODEL)

API key পাওয়া গেছে: True
model: openai/gpt-oss-20b


<a id="s2"></a>

## ধাপ ২: ORDERS ও get_order_status

**ধাপটা কী:** order-এর ছোট data (একটা dict) আর তা থেকে খবর বের করার সাধারণ Python function।
Key হলো order number, value হলো `status` আর `total`।

`ORDERS.get(order_id, not_found)`: key না থাকলে crash না করে `not_found` ফেরত দেয়।

**কেন লাগে:** LLM-কে দেওয়ার আগে function একা ঠিক চলে কিনা দেখে নিতে হয়। Tool আসলে একটা সাধারণ function-ই।

In [32]:
ORDERS = {
    101: {"status": "delivered", "total": 1200},
    102: {"status": "shipped", "total": 850},
    103: {"status": "processing", "total": 430},
}

print("order আছে:", list(ORDERS))

order আছে: [101, 102, 103]


In [33]:
def get_order_status(order_id):
    not_found = {"error": "order not found"}
    return ORDERS.get(order_id, not_found)


print("function তৈরি হয়েছে:", get_order_status.__name__)

function তৈরি হয়েছে: get_order_status


**Test:** 101 আর 102 আছে। 999 নেই, তাই error dict আসবে।

In [34]:
for order_id in [101, 102, 999]:
    print(order_id, "->", get_order_status(order_id))

101 -> {'status': 'delivered', 'total': 1200}
102 -> {'status': 'shipped', 'total': 850}
999 -> {'error': 'order not found'}


<a id="s3"></a>

## ধাপ ৩: INVENTORY ও get_inventory

**ধাপটা কী:** product-এর নাম থেকে stock-এর সংখ্যা। `mouse`-এর stock **0**, মানে product আছে কিন্তু মজুত নেই।

`INVENTORY.get(product_name)` key না পেলে `None` দেয়। তাই `if stock is None` লিখি।
`0` একটা ঠিক মান, `None` মানে product-ই নেই। এই দুটো এক না।

**কেন লাগে:** দ্বিতীয় tool লাগবে, যাতে LLM দুটোর মধ্য থেকে বাছতে পারে।

In [35]:
INVENTORY = {
    "laptop": 12,
    "mouse": 0,
    "keyboard": 5,
}

print("product আছে:", list(INVENTORY))

product আছে: ['laptop', 'mouse', 'keyboard']


In [36]:
def get_inventory(product_name):
    stock = INVENTORY.get(product_name)
    if stock is None:
        return {"error": "product not found"}
    return {"product": product_name, "stock": stock}


print("function তৈরি হয়েছে:", get_inventory.__name__)

function তৈরি হয়েছে: get_inventory


**Test:** `laptop` আছে (12)। `monitor` তালিকায় নেই। `mouse` আছে কিন্তু stock 0।

In [37]:
for product_name in ["laptop", "monitor", "mouse"]:
    print(product_name, "->", get_inventory(product_name))

laptop -> {'product': 'laptop', 'stock': 12}
monitor -> {'error': 'product not found'}
mouse -> {'product': 'mouse', 'stock': 0}


<a id="s4"></a>

## ধাপ ৪: tools list (দুটো tool definition)

**ধাপটা কী:** LLM আমার Python function দেখতে পায় না। তাই JSON schema দিয়ে তাকে বলে দিই প্রতিটা tool-এর:
- `name`: আমার function-এর নাম, হুবহু।
- `description`: কখন এটা লাগবে। LLM এটা পড়েই tool বাছে।
- `parameters`: কোন argument লাগবে (নাম, type, description)।
- `required`: যেগুলো অবশ্যই দিতে হবে।

JSON schema-য় type-এর নাম `integer` আর `string`। Python-এর `int`, `str` না।

**কেন লাগে:** LLM tool সম্পর্কে শুধু এই লেখাটুকুই জানে। Name বা description ভুল হলে সে ভুল tool বাছবে।

<a id="t2"></a>

#### Topic ২: Tool Definition

**আগে একটা গল্প**

দোকানদার কীভাবে জানবেন ছেলেটা কী কী কাজ পারে? ছেলেটা চিনি গুনতে পারে, দোকান ঝাড়ু দিতে পারে, বাজার থেকে দুধ আনতে পারে। দোকানদারকে আগে থেকেই এই কাজের তালিকা দিয়ে রাখতে হয়, নইলে তিনি এমন কাজ করতে বলবেন যা ছেলেটা পারেই না।

**মূল ধারণা**

Request-এর সাথে `tools` নামে একটা তালিকা পাঠানো হয়, সেটাই LLM-এর কাছে কাজের তালিকা। প্রতিটা tool-এর তিনটা অংশ থাকে: `name` (কাজের নাম), `description` (কাজটা কী আর কখন লাগে), আর `parameters` (কাজটা করতে কী কী তথ্য লাগে)। LLM function-এর ভেতরের code কখনো দেখে না। সে শুধু `name` আর `description` পড়ে ঠিক করে কোন tool কখন চাইবে। তাই description খারাপ হলে LLM ভুল tool বেছে নিতে পারে।

**মনে রাখার এক লাইন**

Tool Definition হলো LLM-কে দেওয়া কাজের তালিকা, আর সে শুধু বর্ণনা দেখেই বেছে নেয়।

<a id="t3"></a>

#### Topic ৩: JSON Schema ও Parameter Design

**আগে একটা গল্প**

দোকানদার ছেলেটাকে বললেন, "বাজারে যাও।" ছেলেটা জিজ্ঞেস করল, "কী আনব? কত কেজি?" এই তথ্য না পেলে কাজটা হবে না। তাই কোন কাজের সাথে কী কী তথ্য লাগবে, সেটা আগে থেকে ঠিক করা থাকে, অনেকটা ফর্মের ঘরের মতো। কোন ঘরে কী লেখা যাবে, আর কোন ঘর খালি রাখা যাবে না, সব আগেই বলা থাকে।

**মূল ধারণা**

Tool-এর `parameters` অংশে এই ফর্মটা লেখা হয়, আর এই নিয়মের নাম JSON Schema। `type` বলে মানটা কোন ধরনের (`string`, `integer`), `enum` বলে কোন কোন মান নেওয়া যাবে, আর `required` বলে কোন ঘর বাধ্যতামূলক। `required` না থাকলে LLM অনুমান করে অন্য কোনো মান বসিয়ে দিতে পারে, আর backend-এ argument না এলে function crash করে।

বোঝার জন্য একটা উদাহরণ:

```json
"properties": {
    "city": {"type": "string"},
    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
},
"required": ["city"]
```

এখানে `city` দিতেই হবে, আর `unit` দিলে শুধু `celsius` বা `fahrenheit` হতে পারবে।

**মনে রাখার এক লাইন**

`type`, `enum`, `required` দিয়ে ফর্মের নিয়ম বেঁধে দিলে LLM-এর ভুল কমে।

In [38]:
get_order_status_tool = {
    "type": "function",
    "function": {
        "name": "get_order_status",
        "description": "Get the current status and total price of an order, given its order id.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "integer",
                    "description": "The order number, for example 101",
                },
            },
            "required": ["order_id"],
        },
    },
}

print("tool বানানো হয়েছে:", get_order_status_tool["function"]["name"])

tool বানানো হয়েছে: get_order_status


In [39]:
get_inventory_tool = {
    "type": "function",
    "function": {
        "name": "get_inventory",
        "description": "Get how many units of a product are in stock, given the product name.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {
                    "type": "string",
                    "description": "The product name, for example laptop or mouse",
                },
            },
            "required": ["product_name"],
        },
    },
}

print("tool বানানো হয়েছে:", get_inventory_tool["function"]["name"])

tool বানানো হয়েছে: get_inventory


দুটো tool একই `tools` list-এ রাখি। এই list-ই LLM-কে পাঠানো হবে।

In [40]:
tools = [get_order_status_tool, get_inventory_tool]

for tool in tools:
    function_info = tool["function"]
    print(function_info["name"], "| লাগবে:", function_info["parameters"]["required"])

get_order_status | লাগবে: ['order_id']
get_inventory | লাগবে: ['product_name']


<a id="s5"></a>

## ধাপ ৫: LLM-এর বাছাই দেখা

**ধাপটা কী:** প্রশ্ন আর `tools` একসাথে পাঠিয়ে দেখি LLM কী করে।
- tool লাগলে `message.content` হয় `None`, আর `message.tool_calls`-এ function-এর নাম ও arguments থাকে।
- tool না লাগলে সরাসরি উত্তর আসে, আর `tool_calls` হয় `None`।

**কেন লাগে:** কোন tool কখন লাগবে, সেটা আমি if/else লিখে ঠিক করি না। LLM প্রশ্ন আর description পড়ে নিজেই বাছে।

<a id="t4"></a>

#### Topic ৪: Tool Selection ও Tool Response

**আগে একটা গল্প**

দোকানদারের হাতে এখন দুটো কাজের তালিকা: "গুদামের চিনি গোনা" আর "বাজার থেকে দুধ আনা"। খদ্দের জিজ্ঞেস করল চিনি কত আছে, আর দোকানদার তালিকা দেখে ঠিক কাজটা বেছে নিলেন। ছেলেটা গুনে এসে বলল, "৫০ কেজি।" এই খবর দোকানদারের কাছে না পৌঁছালে খদ্দের কিছুই জানতে পারবে না।

**মূল ধারণা**

Tool Selection হলো `name` ও `description` পড়ে LLM-এর ঠিক tool বেছে নেওয়া। Tool Response হলো function-এর ফলাফল LLM-কে ফেরত দেওয়া। Code-এ এটা `messages`-এ `role: "tool"` দিয়ে পাঠানো হয়, সাথে থাকে `tool_call_id`, যাতে বোঝা যায় এটা কোন কাজের উত্তর। ফলাফল ফেরত না গেলে LLM জানবে না তাপমাত্রা বা stock কত, তাই user-এর জন্য উত্তরও বানাতে পারবে না।

**মনে রাখার এক লাইন**

Model tool বাছে, backend চালায়, আর ফলাফল ফেরত গেলে তবেই model উত্তর দেয়।

In [41]:
question = "আমাদের কাছে laptop কত পিস আছে?"
messages = [{"role": "user", "content": question}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

content: None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='fc_acaa6202-a1e2-43b5-b518-b9d59077dfe2', function=Function(arguments='{"product_name":"laptop"}', name='get_inventory'), type='function')]


আরও কয়েকটা প্রশ্নে LLM কী বাছে দেখি। একটা প্রশ্নে tool-ই লাগে না।

In [42]:
questions = [
    "তোমার নাম কী?",
    "order 102 এর অবস্থা কী?",
    "আমাদের কাছে mouse কত পিস আছে?",
]

for text in questions:
    reply_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
        tools=tools,
    )
    reply = reply_response.choices[0].message
    if reply.tool_calls is None:
        print(text, "-> কোনো tool লাগেনি")
    else:
        print(text, "->", reply.tool_calls[0].function.name)

তোমার নাম কী? -> কোনো tool লাগেনি
order 102 এর অবস্থা কী? -> get_order_status
আমাদের কাছে mouse কত পিস আছে? -> get_inventory


<a id="s6"></a>

## ধাপ ৬: arguments পড়া ও routing

**ধাপটা কী:** `message.tool_calls[0]` থেকে তিনটা জিনিস নিই:
- `tool_call.id`: এই call-এর পরিচয়। পরে result ফেরত দেওয়ার সময় লাগবে।
- `tool_call.function.name`: কোন function চালাতে হবে।
- `tool_call.function.arguments`: argument, কিন্তু এটা **JSON string**, dict না।

String থেকে dict বানাতে `json.loads` (শেষে `s`, মানে string)।

তারপর **routing**: `name` দেখে ঠিক function-এ পাঠানো।

**কেন লাগে:** LLM-এর দেওয়া argument দিয়েই আমার function চলে। ঠিকভাবে না পড়লে tool চালানো যায় না।

In [43]:
tool_call = message.tool_calls[0]
name = tool_call.function.name
args = json.loads(tool_call.function.arguments)

print("arguments-এর type:", type(tool_call.function.arguments).__name__)
print("name:", name)
print("args:", args)

arguments-এর type: str
name: get_inventory
args: {'product_name': 'laptop'}


এবার `name` দেখে function চালাই। কোনোটাই না মিললে error dict।

In [44]:
if name == "get_order_status":
    result = get_order_status(args["order_id"])
elif name == "get_inventory":
    result = get_inventory(args["product_name"])
else:
    result = {"error": "unknown tool"}

print(result)

{'product': 'laptop', 'stock': 12}


<a id="s7"></a>

## ধাপ ৭: result ফেরত দিয়ে চূড়ান্ত উত্তর

**ধাপটা কী:** function-এর result LLM-কে ফেরত দিতে হয়। নইলে সে জানে না কী পাওয়া গেছে।
`messages`-এ তিনটা জিনিস থাকবে:
1. user-এর আসল প্রশ্ন।
2. LLM-এর আগের `message` (যার ভেতরে `tool_calls` আছে)।
3. `role: "tool"` message: `tool_call_id` হলো `tool_call.id`, আর `content` হলো result-এর JSON string (`json.dumps`)।

`messages + [message, tool_message]` লিখলে নতুন list তৈরি হয়। তাই cell আবার চালালে message দ্বিগুণ হয় না।

**কেন লাগে:** এই দ্বিতীয় call-এ LLM কাঁচা JSON-কে সুন্দর বাংলা বাক্যে বদলায়।

In [45]:
tool_message = {
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result, ensure_ascii=False),
}

messages_with_result = messages + [message, tool_message]

print("messages-এ মোট", len(messages_with_result), "টা")
print("tool message:", tool_message)

messages-এ মোট 3 টা
tool message: {'role': 'tool', 'tool_call_id': 'fc_acaa6202-a1e2-43b5-b518-b9d59077dfe2', 'content': '{"product": "laptop", "stock": 12}'}


In [46]:
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages_with_result,
    tools=tools,
)

print(final_response.choices[0].message.content)

আমাদের কাছে ল্যাপটপের ১২ পিস স্টকে আছে।


<a id="s8"></a>

## ধাপ ৮: run_tool function

**ধাপটা কী:** ধাপ ৬-এর routing-কে একটা function-এ ভরা: `run_tool(name, args)`।
এখন যতগুলো tool-ই থাকুক, এক লাইনে `run_tool(name, args)` ডাকলেই হয়।

নাম না মিললে `{"error": "unknown tool"}` ফেরত আসে।

**কেন লাগে:** LLM কখনো অচেনা বা ভুল নাম বললেও program ভাঙে না। শুধু আমার লিখে দেওয়া function-ই চলে, অন্য কিছু না।

In [47]:
def run_tool(name, args):
    if name == "get_order_status":
        return get_order_status(args["order_id"])
    if name == "get_inventory":
        return get_inventory(args["product_name"])
    return {"error": "unknown tool"}


print("function তৈরি হয়েছে:", run_tool.__name__)

function তৈরি হয়েছে: run_tool


**Test:** `mouse` চেনা tool, stock 0। `delete_everything` অচেনা, তাই কিছুই চলবে না।

In [48]:
tests = [
    ("get_inventory", {"product_name": "mouse"}),
    ("delete_everything", {}),
]

for test_name, test_args in tests:
    print(test_name, "->", run_tool(test_name, test_args))

get_inventory -> {'product': 'mouse', 'stock': 0}
delete_everything -> {'error': 'unknown tool'}


<a id="s9"></a>

## ধাপ ৯: দুই অংশের প্রশ্ন (sequential tool calls)

**ধাপটা কী:** একটা প্রশ্নে দুটো জিনিস জিজ্ঞেস করলে (order আর laptop), LLM এক রাউন্ডে শুধু **একটা** tool চায়।
আমার code যদি শুধু `tool_calls[0]` সামলায়, তাহলে দ্বিতীয় অংশের উত্তর আসে না।

সমাধান: একটা loop। প্রতি রাউন্ডে
1. LLM-কে call করি।
2. `tool_calls` `None` হলে চূড়ান্ত উত্তর পেয়েছি, থামি।
3. না হলে প্রতিটা call চালিয়ে result `messages`-এ যোগ করি, তারপর আবার রাউন্ড।

এভাবে tool একটার পর একটা চলে। একে sequential tool calls বলে।

**কেন লাগে:** আসল agent এভাবেই কাজ করে: কাজ শেষ না হওয়া পর্যন্ত tool চায়, চালায়, আবার জিজ্ঞেস করে।

In [49]:
question_two = "order 102 এর অবস্থা কী, আর laptop কত পিস আছে?"

first_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question_two}],
    tools=tools,
)
first_message = first_response.choices[0].message

print("tool_calls-এর সংখ্যা:", len(first_message.tool_calls))
for call in first_message.tool_calls:
    print(call.function.name, call.function.arguments)

tool_calls-এর সংখ্যা: 1
get_order_status {"order_id":102}


এবার loop দিয়ে সব tool সামলাই। বেশি হলে ৩ রাউন্ড, তারপর থামি।

In [50]:
conversation = [{"role": "user", "content": question_two}]

for round_number in range(1, 4):
    round_response = client.chat.completions.create(
        model=MODEL,
        messages=conversation,
        tools=tools,
    )
    round_message = round_response.choices[0].message

    if round_message.tool_calls is None:
        print("চূড়ান্ত উত্তর:", round_message.content)
        break

    conversation = conversation + [round_message]
    for call in round_message.tool_calls:
        call_args = json.loads(call.function.arguments)
        call_result = run_tool(call.function.name, call_args)
        print("রাউন্ড", round_number, "|", call.function.name, call_args, "->", call_result)
        tool_reply = {
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(call_result, ensure_ascii=False),
        }
        conversation = conversation + [tool_reply]
else:
    print("৩ রাউন্ডেও চূড়ান্ত উত্তর আসেনি")

রাউন্ড 1 | get_order_status {'order_id': 102} -> {'status': 'shipped', 'total': 850}
রাউন্ড 2 | get_inventory {'product_name': 'laptop'} -> {'product': 'laptop', 'stock': 12}
চূড়ান্ত উত্তর: অর্ডার **102** এখন **শিপড** (ডেলিভারি করা হয়েছে) এবং মোট মূল্য **₹৮৫০**।

**ল্যাপটপ** ইনভেন্টরি: বর্তমানে **১২টি পিস** স্টকে আছে।


<a id="s10"></a>

## ধাপ ১০: আমার করা ভুলগুলো ও self-check

<a id="s11"></a>

### আমার করা ভুলগুলো

এই ভুলগুলো আমি করেছিলাম। উপরের ধাপগুলোয় এগুলো নেই, শুধু ঠিক code আছে।

| # | আমার ভুল | কেন হয়েছিল | ঠিক করার উপায় |
|---|---|---|---|
| 1 | **NameError:** `tools=tools` লিখলাম, অথচ `tools` তখনো বানানো হয়নি। Error: `NameError: name 'tools' is not defined` | আমি cell গুলো ইচ্ছেমতো ক্রমে চালিয়েছিলাম। কোনো নাম বানানোর আগেই সেটা ব্যবহার করেছি | উপর থেকে ক্রমে চালাও। Error-এর শেষ লাইনে যে নাম আছে, সেটা কোন cell-এ বানানো হয় খুঁজে সেই cell আগে চালাও |
| 2 | **json.load:** `json.load(tool_call.function.arguments)`। Error: `AttributeError: 'str' object has no attribute 'read'` | `load` file পড়ে। কিন্তু `arguments` একটা string, file না | string-এর জন্য `json.loads(tool_call.function.arguments)`। মনে রাখার উপায়: `s` মানে string |
| 3 | **set বনাম dict:** `INVENTORY = {"laptop", 12, "mouse", 0}`। Error: `AttributeError: 'set' object has no attribute 'get'` | `key: value` লিখিনি, শুধু comma দিয়েছি। তখন সেটা dict না, set হয়ে যায় | প্রতিটা key-র পর `:` দাও: `{"laptop": 12, "mouse": 0}` |
| 4 | **0 কে missing ধরা:** `if not stock:` | `0` কেও `not` দিলে true হয়। তাই `mouse` (stock 0) "product not found" দেখাত | `if stock is None:`। `None` মানে key নেই, `0` মানে আছে কিন্তু মজুত নেই |
| 5 | **schema-য় placeholder:** `"description": "TODO"` বা `"description here"` বা অন্য tool থেকে কপি করা পুরোনো লেখা | পরে ঠিক করব ভেবে রেখে দিয়েছিলাম। কিন্তু LLM description পড়েই tool বাছে | প্রতিটা tool-এ পরিষ্কার এক বাক্যে লেখো সে কী করে। Argument-এর description-এ একটা উদাহরণ দাও |
| 6 | **restart করে আগের cell না চালানো:** kernel restart করে সরাসরি নিচের cell চালালাম | Restart-এ সব variable মুছে যায় (যেমন `client`, `tools`, `ORDERS`)। ফল আবার `NameError` | restart-এর পর **Run All** করো, অথবা অন্তত ধাপ ১ থেকে ক্রমে চালাও |

<a id="s12"></a>

### Self-check

নিজে উত্তর দিয়ে তারপর নিচে মিলিয়ে নাও।

**Function calling**
1. LLM কি নিজে function চালায়? তাহলে কে চালায়?
2. পুরো flow-র পাঁচ ধাপ ক্রমে বলো।

**Function আর tool definition**
3. `get_inventory`-তে `is None` কেন, `if not stock` কেন না?
4. Tool definition-এর `name`, `description`, `parameters`, `required` প্রতিটা কী কাজে লাগে?
5. `INVENTORY = {"laptop", 12}` লিখলে কী হয়?

**LLM-এর বাছাই আর arguments**
6. কোনো tool না লাগলে `content` আর `tool_calls`-এ কী থাকে?
7. `tool_call.function.arguments` কোন type-এর? `json.load` আর `json.loads`-এর পার্থক্য কী?
8. `tool_call.id` কেন লাগে?

**Routing আর চূড়ান্ত উত্তর**
9. `run_tool("delete_everything", {})` কী ফেরত দেয়? এটা কেন ভালো?
10. Result ফেরত দেওয়ার `messages`-এ কোন তিনটা জিনিস থাকে? `json.dumps` কেন?
11. দুই অংশের প্রশ্নে LLM এক রাউন্ডে কয়টা tool চায়? সবগুলো সামলাতে কী করতে হয়?
12. Notebook restart করার পর `NameError` এলে কী করব?

<details>
<summary><b>উত্তর (আগে নিজে ভাবো, তারপর খোলো)</b></summary>

1. **না।** LLM শুধু বলে কোন function কোন argument দিয়ে চালাতে হবে। চালায় **আমার code**।
2. প্রশ্ন ও `tools` পাঠাই → LLM `tool_calls` দেয় → আমার code function চালায় → result `role: "tool"` message-এ ফেরত দিই → LLM চূড়ান্ত উত্তর লেখে।
3. `0` একটা ঠিক মান (product আছে, stock নেই)। `if not stock` লিখলে `0`-ও "নেই" ধরা পড়ে। `is None` শুধু key না থাকলে true হয়।
4. `name`: আমার function-এর নাম, হুবহু। `description`: কখন লাগবে, LLM এটা পড়ে বাছে। `parameters`: argument-এর নাম, type, description। `required`: যেগুলো অবশ্যই দিতে হবে।
5. `key: value` নেই, তাই এটা **set**, dict না। `.get` ডাকলে `AttributeError`।
6. `content`-এ সরাসরি উত্তর থাকে, `tool_calls` হয় `None`।
7. **string** (JSON string)। `json.load` file থেকে পড়ে, `json.loads` string থেকে। এখানে `json.loads`।
8. Result ফেরত দেওয়ার সময় `tool_call_id` দিতে হয়, যাতে LLM বোঝে এটা কোন call-এর result।
9. `{'error': 'unknown tool'}`। অচেনা নামে কিছু চলে না, program-ও ভাঙে না।
10. user-এর প্রশ্ন, LLM-এর আগের `message`, আর `role: "tool"` message। `content` string হতে হয়, তাই dict-কে `json.dumps` দিয়ে string বানাই।
11. সাধারণত **একটা**। সবগুলো সামলাতে loop লাগে: প্রতি রাউন্ডে tool চালাই, result যোগ করি, `tool_calls` `None` না হওয়া পর্যন্ত আবার call করি।
12. উপর থেকে Run All করব। কারণ restart-এ সব variable মুছে গেছে।

</details>